# Exercícios Práticos: KNN e SVM - Detecção de Spam

**MPTI - Aprendizado de Máquina 2026.2**  
**Prof. Diego Pessoa | Prof. Thiago Moura**  
**IFPB - Instituto Federal da Paraíba**

---

## Objetivo
Implementar e comparar os algoritmos **KNN** e **SVM Linear** para detecção de spam em emails.

## Problema
📧 **Filtro de Spam**: Classificar emails como spam (1) ou não-spam (0) baseado no conteúdo do texto.

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Configurar matplotlib para Jupyter
plt.style.use('default')
%matplotlib inline

np.random.seed(42)

## Criação do Dataset

In [2]:
# Palavras típicas de SPAM
spam_words = ['gratis', 'oferta', 'promocao', 'urgente', 'dinheiro', 'ganhe', 'clique', 'premio']

# Palavras típicas de emails NORMAIS
normal_words = ['reuniao', 'trabalho', 'projeto', 'familia', 'amigos', 'estudo', 'saude', 'viagem']

def gerar_email(tipo):
    """Gera um email sintético"""
    if tipo == 'spam':
        # SPAM: 70% palavras spam + 30% palavras normais (para confundir)
        n_spam = np.random.randint(4, 6)
        n_normal = np.random.randint(1, 3)
        palavras_spam = np.random.choice(spam_words, size=n_spam, replace=True)
        palavras_normal = np.random.choice(normal_words, size=n_normal, replace=True)
        palavras = np.concatenate([palavras_spam, palavras_normal])
    else:
        # NORMAL: 80% palavras normais + 20% palavras spam (para confundir)
        n_normal = np.random.randint(4, 7)
        n_spam = np.random.randint(0, 2)  # Às vezes 0 palavras spam
        palavras_normal = np.random.choice(normal_words, size=n_normal, replace=True)
        if n_spam > 0:
            palavras_spam = np.random.choice(spam_words, size=n_spam, replace=True)
            palavras = np.concatenate([palavras_normal, palavras_spam])
        else:
            palavras = palavras_normal

    np.random.shuffle(palavras)
    return ' '.join(palavras)

# Gerando 1000 emails (500 spam + 500 normais) - dataset maior
emails = []

for i in range(500):
    emails.append({'texto': gerar_email('spam'), 'label': 1})    # spam = 1
    emails.append({'texto': gerar_email('normal'), 'label': 0})  # normal = 0

df = pd.DataFrame(emails)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # embaralhar

print(f"Dataset criado: {len(df)} emails")
print(f"- Spam: {df['label'].sum()} emails")
print(f"- Normal: {len(df) - df['label'].sum()} emails")

print("\nExemplos:")
print(f"SPAM: {df[df['label']==1]['texto'].iloc[0]}")
print(f"NORMAL: {df[df['label']==0]['texto'].iloc[0]}")

# Mostrar alguns exemplos que podem confundir o modelo
print("\nExemplos que podem confundir:")
spam_examples = df[df['label']==1]['texto'].head(3)
normal_examples = df[df['label']==0]['texto'].head(3)
for i, email in enumerate(spam_examples, 1):
    print(f"SPAM {i}: {email}")
for i, email in enumerate(normal_examples, 1):
    print(f"NORMAL {i}: {email}")

Dataset criado: 1000 emails
- Spam: 500 emails
- Normal: 500 emails

Exemplos:
SPAM: promocao urgente urgente oferta trabalho urgente
NORMAL: estudo saude viagem projeto

Exemplos que podem confundir:
SPAM 1: promocao urgente urgente oferta trabalho urgente
SPAM 2: promocao premio premio gratis clique saude
SPAM 3: premio estudo promocao ganhe ganhe
NORMAL 1: estudo saude viagem projeto
NORMAL 2: reuniao viagem reuniao urgente amigos saude
NORMAL 3: trabalho estudo clique viagem viagem amigos familia


## Pré-processamento

In [3]:
# Pré-processamento dos dados
vectorizer = TfidfVectorizer(max_features=50)  # Reduzindo features para mais desafio
X = vectorizer.fit_transform(df['texto'])
y = df['label'].values

print(f"✅ Textos convertidos para vetores: {X.shape}")
print(f"✅ Vocabulário: {len(vectorizer.get_feature_names_out())} palavras")

# Divisão dos dados em treino e teste (mais treino para melhor aprendizado)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

print(f"\n📊 Divisão dos dados:")
print(f"- Treino: {X_train.shape[0]} emails")
print(f"- Teste: {X_test.shape[0]} emails")
print(f"- Distribuição treino - Spam: {y_train.sum()}, Normal: {len(y_train) - y_train.sum()}")
print(f"- Distribuição teste - Spam: {y_test.sum()}, Normal: {len(y_test) - y_test.sum()}")

✅ Textos convertidos para vetores: (1000, 16)
✅ Vocabulário: 16 palavras

📊 Divisão dos dados:
- Treino: 600 emails
- Teste: 400 emails
- Distribuição treino - Spam: 300, Normal: 300
- Distribuição teste - Spam: 200, Normal: 200


---

# 🔵 EXERCÍCIO 1: KNN para Detecção de Spam

O KNN classifica baseado nos K vizinhos mais próximos.

## 1.1 Normalização dos Dados

In [4]:
# Normalização dos dados para KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.toarray())
X_test_scaled = scaler.transform(X_test.toarray())

print("Dados normalizados para o KNN")

Dados normalizados para o KNN


## 1.2 Encontrando o Melhor K

In [5]:
# EXERCÍCIO 1: Configure o melhor valor de K para o KNN
# Testa diferentes valores de K e escolhe o melhor com base na acurácia média

k_values = [1, 3, 5, 7, 9]
acc_by_k = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    acc_by_k.append((k, scores.mean()))
    print(f"K={k}: acurácia média = {scores.mean():.4f}")

best_k = min(k_values, key=lambda k: (-cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_scaled, y_train, cv=5).mean(), k))

print(f"\nValor de K escolhido: {best_k}")

K=1: acurácia média = 1.0000
K=3: acurácia média = 1.0000
K=5: acurácia média = 1.0000
K=7: acurácia média = 1.0000
K=9: acurácia média = 1.0000

Valor de K escolhido: 1


## 1.3 Avaliação Final do KNN

In [6]:
# Treinar o KNN final
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)

print("✅ KNN treinado com sucesso!")

✅ KNN treinado com sucesso!


---

# 🔴 EXERCÍCIO 2: SVM Linear para Detecção de Spam

O SVM busca o hiperplano que maximiza a margem entre as classes.

## 2.1 Testando Diferentes Valores de C

In [7]:
# EXERCÍCIO 2: Configure o melhor valor de C para o SVM
# Testa diferentes valores de C e escolhe o melhor com base na acurácia média

C_values = [0.1, 1, 10, 100]
acc_by_c = []

for c in C_values:
    svm = SVC(kernel='linear', C=c)
    scores = cross_val_score(svm, X_train, y_train, cv=5)
    acc_by_c.append((c, scores.mean()))
    print(f"C={c}: acurácia média = {scores.mean():.4f}")

best_C = min(C_values, key=lambda c: (-cross_val_score(SVC(kernel='linear', C=c), X_train, y_train, cv=5).mean(), c))

print(f"\nValor de C escolhido: {best_C}")

C=0.1: acurácia média = 1.0000
C=1: acurácia média = 1.0000
C=10: acurácia média = 1.0000
C=100: acurácia média = 1.0000

Valor de C escolhido: 0.1


## 2.2 Avaliação Final do SVM

In [8]:
# Treinar o SVM final
final_svm = SVC(kernel='linear', C=best_C)
final_svm.fit(X_train, y_train)

print("✅ SVM treinado com sucesso!")

✅ SVM treinado com sucesso!


## 2.3 Interpretabilidade do SVM

In [10]:
## Palavras Mais Importantes do SVM

# Analisando quais palavras são mais importantes para classificação
feature_names = vectorizer.get_feature_names_out()
coef_raw = final_svm.coef_
if hasattr(coef_raw, 'toarray'):
    coef = coef_raw.toarray().ravel()
else:
    coef = np.asarray(coef_raw).ravel()

# Top 5 palavras que mais indicam SPAM (coeficientes mais positivos)
spam_indices = np.argsort(coef)[-5:][::-1]
print("🚨 Top 5 palavras que indicam SPAM:")
for i in spam_indices:
    print(f"   '{feature_names[i]}': {coef[i]:.3f}")

# Top 5 palavras que mais indicam NORMAL (coeficientes mais negativos)
normal_indices = np.argsort(coef)[:5]
print("\n✅ Top 5 palavras que indicam EMAIL NORMAL:")
for i in normal_indices:
    print(f"   '{feature_names[i]}': {coef[i]:.3f}")

🚨 Top 5 palavras que indicam SPAM:
   'gratis': 1.039
   'oferta': 1.005
   'ganhe': 0.983
   'dinheiro': 0.971
   'clique': 0.932

✅ Top 5 palavras que indicam EMAIL NORMAL:
   'estudo': -0.873
   'amigos': -0.864
   'reuniao': -0.825
   'saude': -0.822
   'viagem': -0.812


In [11]:
# 🧪 TESTE COM NOVOS EMAILS

# Testando com emails novos - o foco principal do exercício
emails_teste = [
    "reuniao trabalho projeto",
    "oferta gratis dinheiro clique",
    "familia amigos jantar",
    "promocao urgente ganhe premio",
    "estudo universidade pesquisa",
    "clique aqui ganhe muito dinheiro"
]

print("🧪 Testando classificação de emails novos:")
print("=" * 50)

for email in emails_teste:
    email_vector = vectorizer.transform([email])
    email_scaled = scaler.transform(email_vector.toarray())

    knn_pred = final_knn.predict(email_scaled)[0]
    svm_pred = final_svm.predict(email_vector)[0]

    knn_label = "SPAM" if knn_pred == 1 else "NORMAL"
    svm_label = "SPAM" if svm_pred == 1 else "NORMAL"

    print(f"📧 '{email}'")
    print(f"   KNN: {knn_label}")
    print(f"   SVM: {svm_label}")
    print()


🧪 Testando classificação de emails novos:
📧 'reuniao trabalho projeto'
   KNN: NORMAL
   SVM: NORMAL

📧 'oferta gratis dinheiro clique'
   KNN: SPAM
   SVM: SPAM

📧 'familia amigos jantar'
   KNN: NORMAL
   SVM: NORMAL

📧 'promocao urgente ganhe premio'
   KNN: SPAM
   SVM: SPAM

📧 'estudo universidade pesquisa'
   KNN: NORMAL
   SVM: NORMAL

📧 'clique aqui ganhe muito dinheiro'
   KNN: SPAM
   SVM: SPAM



---

# 🧪 EXERCÍCIO PRINCIPAL: Classificação de Novos Emails

Agora vamos focar no objetivo principal: classificar emails novos usando os algoritmos treinados.

## Teste com Novos Emails

In [12]:
# 📝 EXERCÍCIO EXTRA: Teste seus próprios emails

print("📝 Agora teste com seus próprios emails!")
print("Adicione novos emails na lista abaixo e veja como os algoritmos classificam:")

meus_emails = [
    "compre agora com desconto",
    "vamos almocar amanha",
    "oferta gratis dinheiro clique",
    "reuniao trabalho projeto",
]

print("\n🧪 Classificando seus emails:")
print("=" * 50)
for email in meus_emails:
    email_vector = vectorizer.transform([email])
    email_scaled = scaler.transform(email_vector.toarray())

    knn_pred = final_knn.predict(email_scaled)[0]
    svm_pred = final_svm.predict(email_vector)[0]

    knn_label = "SPAM" if knn_pred == 1 else "NORMAL"
    svm_label = "SPAM" if svm_pred == 1 else "NORMAL"

    print(f"📧 '{email}'")
    print(f"   KNN: {knn_label}")
    print(f"   SVM: {svm_label}")
    print()

print("✅ Teste concluído. Edite a lista `meus_emails` e execute novamente.")

# TODO: Use o mesmo código de classificação acima para testar seus emails
# Dica: copie o código do loop anterior e modifique para usar meus_emails

📝 Agora teste com seus próprios emails!
Adicione novos emails na lista abaixo e veja como os algoritmos classificam:

🧪 Classificando seus emails:
📧 'compre agora com desconto'
   KNN: NORMAL
   SVM: SPAM

📧 'vamos almocar amanha'
   KNN: NORMAL
   SVM: SPAM

📧 'oferta gratis dinheiro clique'
   KNN: SPAM
   SVM: SPAM

📧 'reuniao trabalho projeto'
   KNN: NORMAL
   SVM: NORMAL

✅ Teste concluído. Edite a lista `meus_emails` e execute novamente.


---

# 🤔 QUESTÕES PARA REFLEXÃO

**1. Que valor de K você escolheu para o KNN? Por quê?**
_Sua resposta:_

**2. Que valor de C você escolheu para o SVM? Por quê?**
_Sua resposta:_

**3. Qual algoritmo classificou melhor os emails de teste? KNN ou SVM?**
_Sua resposta:_

**4. Por que você acha que o KNN precisa de normalização dos dados?**
_Sua resposta:_

**5. Como você poderia melhorar este sistema de detecção de spam?**
_Sua resposta:_

---

# ✅ RESUMO

## O que você fez neste exercício:
✅ **Configurou** o valor de K para o algoritmo KNN  
✅ **Configurou** o valor de C para o algoritmo SVM Linear  
✅ **Treinou** ambos os algoritmos com dados de exemplo  
✅ **Testou** a classificação de novos emails  
✅ **Comparou** os resultados dos dois algoritmos  

## Conceitos importantes:
- **KNN**: Precisa de normalização, sensível ao valor de K
- **SVM**: Busca a melhor margem, interpretável através dos coeficientes
- **Pré-processamento**: TF-IDF transforma texto em números
- **Teste**: Importante validar com dados novos

---

**🎓 Prof. Diego Pessoa | Prof. Thiago Moura**  
**🏫 IFPB - Instituto Federal da Paraíba**